In [ ]:
# 1. SỬ DỤNG THƯ VIỆN
# mlxtend cung cấp thuật toán FP-Growth để khai phá các tập mục phổ biến.
%pip install -q mlxtend pandas matplotlib


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth

RANDOM_STATE = 42


In [ ]:
# 2. TỰ CÀI ĐẶT THUẬT TOÁN FP-GROWTH
# Mỗi nút lưu tên item, số lần xuất hiện, nút cha và các nút cùng item trong header table.

class FPNode:
    def __init__(self, item, count, parent):
        self.item = item
        self.count = count
        self.parent = parent
        self.children = {}
        self.node_link = None


# Header table quản lý tổng support và liên kết tất cả nút mang cùng một item.
class HeaderEntry:
    def __init__(self, support):
        self.support = support
        self.head = None
        self.tail = None

    def add_node(self, node):
        if self.head is None:
            self.head = node
        else:
            self.tail.node_link = node
        self.tail = node


def build_fp_tree(weighted_transactions, min_support_count):
    # Đếm support của từng item từ các giao dịch có trọng số.
    item_support = {}
    for transaction, weight in weighted_transactions:
        for item in set(transaction):
            item_support[item] = item_support.get(item, 0) + weight

    # Loại item không đạt minimum support.
    frequent_items = {
        item: count
        for item, count in item_support.items()
        if count >= min_support_count
    }

    if not frequent_items:
        return None, {}

    header = {
        item: HeaderEntry(count)
        for item, count in frequent_items.items()
    }

    root = FPNode(None, 0, None)

    # Sắp xếp item theo support giảm dần để tạo các prefix path dùng chung.
    for transaction, weight in weighted_transactions:
        filtered = [item for item in transaction if item in frequent_items]
        filtered.sort(key=lambda item: (-frequent_items[item], str(item)))

        current = root
        for item in filtered:
            if item in current.children:
                child = current.children[item]
                child.count += weight
            else:
                child = FPNode(item, weight, current)
                current.children[item] = child
                header[item].add_node(child)
            current = child

    return root, header


def conditional_pattern_base(item, header_entry):
    # Truy ngược từ từng node của item về root để lấy prefix path.
    pattern_base = []
    node = header_entry.head

    while node is not None:
        path = []
        parent = node.parent

        while parent is not None and parent.item is not None:
            path.append(parent.item)
            parent = parent.parent

        if path:
            pattern_base.append((path, node.count))

        node = node.node_link

    return pattern_base


def mine_fp_tree(header, min_support_count, prefix, frequent_patterns):
    # Khai phá từ item có support thấp lên cao để sinh các frequent itemsets.
    ordered_items = sorted(
        header.keys(),
        key=lambda item: (header[item].support, str(item))
    )

    for item in ordered_items:
        new_pattern = prefix | frozenset([item])
        frequent_patterns[new_pattern] = header[item].support

        # Xây dựng conditional pattern base và conditional FP-tree.
        conditional_base = conditional_pattern_base(item, header[item])
        conditional_tree, conditional_header = build_fp_tree(
            conditional_base,
            min_support_count
        )

        if conditional_tree is not None:
            mine_fp_tree(
                conditional_header,
                min_support_count,
                new_pattern,
                frequent_patterns
            )


def fp_growth(transactions, min_support=0.2):
    # Hàm chính: chuyển dữ liệu thành giao dịch có trọng số, dựng FP-tree và khai phá.
    if not transactions:
        return {}

    min_support_count = int(np.ceil(min_support * len(transactions)))
    weighted_transactions = [(list(set(t)), 1) for t in transactions]

    tree, header = build_fp_tree(weighted_transactions, min_support_count)

    frequent_patterns = {}
    if tree is not None:
        mine_fp_tree(header, min_support_count, frozenset(), frequent_patterns)

    # Chuyển support count thành support ratio để dễ so sánh với mlxtend.
    return {
        itemset: count / len(transactions)
        for itemset, count in frequent_patterns.items()
    }


def format_itemsets(patterns):
    # Chuyển itemset dạng frozenset thành chuỗi để hiển thị trong DataFrame.
    rows = [
        {"itemsets": tuple(sorted(itemset)), "support": support}
        for itemset, support in patterns.items()
    ]
    return pd.DataFrame(rows).sort_values(
        ["support", "itemsets"],
        ascending=[False, True]
    ).reset_index(drop=True)


In [ ]:
# 3. DATA MOCK VÀ KIỂM THỬ HAI THUẬT TOÁN
# Tạo dữ liệu giao dịch mua hàng với một số nhóm sản phẩm thường xuất hiện cùng nhau.

transactions = [
    ["Bread", "Milk"],
    ["Bread", "Diaper", "Beer", "Eggs"],
    ["Milk", "Diaper", "Beer", "Cola"],
    ["Bread", "Milk", "Diaper", "Beer"],
    ["Bread", "Milk", "Diaper", "Cola"],
    ["Bread", "Milk", "Eggs"],
    ["Milk", "Diaper", "Beer"],
    ["Bread", "Diaper", "Cola"],
    ["Bread", "Milk", "Diaper", "Beer", "Eggs"],
    ["Milk", "Beer", "Cola"],
    ["Bread", "Milk", "Diaper"],
    ["Bread", "Beer", "Eggs"],
    ["Milk", "Diaper", "Cola"],
    ["Bread", "Milk", "Beer"],
    ["Bread", "Diaper", "Beer", "Cola"],
    ["Milk", "Eggs"],
    ["Bread", "Milk", "Diaper", "Beer"],
    ["Bread", "Cola"],
    ["Milk", "Diaper", "Beer", "Eggs"],
    ["Bread", "Milk", "Diaper", "Cola"]
]

transaction_df = pd.DataFrame({
    "Transaction_ID": range(1, len(transactions) + 1),
    "Items": transactions
})
display(transaction_df)
print("Số giao dịch:", len(transactions))


In [ ]:
# Chọn minimum support chung để hai thuật toán sử dụng cùng tiêu chí.
MIN_SUPPORT = 0.20
print("Minimum support:", MIN_SUPPORT)
print("Minimum support count:", int(np.ceil(MIN_SUPPORT * len(transactions))))


In [ ]:
# 3.1. Chạy FP-Growth bằng thư viện mlxtend.
# TransactionEncoder chuyển danh sách giao dịch thành ma trận one-hot.
encoder = TransactionEncoder()
encoded_array = encoder.fit(transactions).transform(transactions)
one_hot_df = pd.DataFrame(encoded_array, columns=encoder.columns_)

library_result = fpgrowth(
    one_hot_df,
    min_support=MIN_SUPPORT,
    use_colnames=True
)

library_patterns = {
    frozenset(itemset): float(support)
    for itemset, support in zip(
        library_result["itemsets"],
        library_result["support"]
    )
}

display(format_itemsets(library_patterns))
print("Số frequent itemsets từ mlxtend:", len(library_patterns))


In [ ]:
# 3.2. Chạy FP-Growth tự cài đặt.
custom_patterns = fp_growth(transactions, min_support=MIN_SUPPORT)

display(format_itemsets(custom_patterns))
print("Số frequent itemsets tự cài đặt:", len(custom_patterns))


In [ ]:
# 3.3. So sánh hai kết quả theo itemset và support.
all_itemsets = set(library_patterns) | set(custom_patterns)

comparison_rows = []
for itemset in all_itemsets:
    lib_support = library_patterns.get(itemset)
    custom_support = custom_patterns.get(itemset)

    comparison_rows.append({
        "Itemset": tuple(sorted(itemset)),
        "Library_Support": lib_support,
        "Custom_Support": custom_support,
        "Match": (
            lib_support is not None
            and custom_support is not None
            and np.isclose(lib_support, custom_support, atol=1e-12)
        )
    })

comparison_df = pd.DataFrame(comparison_rows).sort_values(
    "Itemset"
).reset_index(drop=True)

display(comparison_df)

print("Tổng itemset:", len(all_itemsets))
print("Itemset trùng khớp:", comparison_df["Match"].sum())
print("Hai kết quả giống nhau:",
      len(library_patterns) == len(custom_patterns)
      and comparison_df["Match"].all())


In [ ]:
# 3.4. Trực quan hóa support của các frequent itemsets.
# Chỉ hiển thị một số itemset có support cao nhất để biểu đồ dễ đọc.
plot_df = format_itemsets(library_patterns).head(15).copy()
plot_df["Itemset"] = plot_df["itemsets"].apply(
    lambda items: ", ".join(items)
)

plt.figure(figsize=(10, 6))
plt.barh(plot_df["Itemset"][::-1], plot_df["support"][::-1])
plt.xlabel("Support")
plt.ylabel("Frequent itemset")
plt.title("Top frequent itemsets - FP-Growth")
plt.tight_layout()
plt.show()


In [ ]:
# 3.5. Kiểm thử thay đổi minimum support.
# Support càng cao thì điều kiện xuất hiện càng nghiêm ngặt và số itemset thường giảm.
support_thresholds = [0.15, 0.20, 0.30, 0.40]
threshold_results = []

for threshold in support_thresholds:
    lib_count = len(fpgrowth(
        one_hot_df,
        min_support=threshold,
        use_colnames=True
    ))

    custom_count = len(fp_growth(
        transactions,
        min_support=threshold
    ))

    threshold_results.append({
        "Min_Support": threshold,
        "Library_Count": lib_count,
        "Custom_Count": custom_count
    })

threshold_df = pd.DataFrame(threshold_results)
display(threshold_df)


In [ ]:
# 3.6. Kiểm tra một số frequent itemsets cụ thể.
# Có thể thay đổi itemset cần kiểm tra để xem support trong kết quả khai phá.
items_to_check = ["Bread", "Milk"]

target_itemset = frozenset(items_to_check)
print("Itemset cần kiểm tra:", items_to_check)
print("Support từ thư viện:", library_patterns.get(target_itemset, 0))
print("Support tự cài đặt:", custom_patterns.get(target_itemset, 0))
